<a href="https://colab.research.google.com/github/suphakornwong/Chapter_4/blob/main/ObjectDetection_Train_Method.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Things to Prepare Before Coding
1. Yolo Pretraind model downloaded from https://docs.ultralytics.com/tasks/detect
2. API key form Roboflow platform https://roboflow.com/

Code Cell-A

In [1]:
!pip install ultralytics --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 10.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics]


Code Cell-B

In [ ]:
import cv2
import torch
import torchvision

print(cv2.__version__, torch.__version__, torchvision.__version__, torch.cuda.is_available())

Code Cell-C

In [ ]:
import os
HOME = os.getcwd()
print(HOME)
%mkdir -p {HOME}/datasets
%cd {HOME}/datasets


Code Cell-D   # Upload from your computer

In [ ]:
!unzip /content/PestDataBase.zip

Code Cell-E    ## This cell is API key from Roboflow platform

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="XXXXXXXXXXXXXXXXXXXXXXXXX")
project = rf.workspace("phuwadorn-oeu02").project("argi_bird")
version = project.version(1)
dataset = version.download("yolo26")


Code Cell-F   # Save your result to google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
folder_path = '/content/drive/MyDrive/Detection'
os.makedirs(folder_path, exist_ok=True)
print(f"Folder created at: {folder_path}")
save_dir = '/content/drive/MyDrive/Detection'


Code Cell-G

In [ ]:
import shutil
import random
from tqdm import tqdm

def split_dataset(dataset_dir, output_dir, train_ratio=0.75, val_ratio=0.15, test_ratio=0.1, seed=42):
   assert train_ratio + val_ratio + test_ratio == 1.0, "Ratios must sum to 1"

   random.seed(seed)
   classes = os.listdir(dataset_dir)

   for class_name in classes:
       class_dir = os.path.join(dataset_dir, class_name)
       if not os.path.isdir(class_dir):
           continue

       images = os.listdir(class_dir)
       random.shuffle(images)

       total = len(images)
       train_count = int(train_ratio * total)
       val_count = int(val_ratio * total)

       train_files = images[:train_count]
       val_files = images[train_count:train_count + val_count]
       test_files = images[train_count + val_count:]

       # Copy files
       for split, split_files in zip(['train', 'val', 'test'], [train_files, val_files, test_files]):
           split_class_dir = os.path.join(output_dir, split, class_name) # Removed the extra 'data' subdirectory
           os.makedirs(split_class_dir, exist_ok=True)

           for file in tqdm(split_files, desc=f"Copying {split}/{class_name}"):
               src = os.path.join(class_dir, file)
               dst = os.path.join(split_class_dir, file)
               if os.path.isfile(src):  # Add this check
                   shutil.copy2(src, dst)


Code Cell-(I-D)

In [ ]:
from ultralytics import YOLO

model = YOLO('/content/yolov8n.pt')      ## Right-click at Pretraind model (for example, yolo8n.pt) in /content, Select "copy path"
model.train(
 data='/content/datasets/Pigeon_main-6/data.yaml',    ## find "data.yaml" in datasets folder, Right-click and Select "copy path"
 task='detect',
 epochs=100,
 imgsz=640,
 device=0
)